<a href="https://colab.research.google.com/github/gaeun1961/DS_Basic_Analysis_Python/blob/main/Ch10_02_%EB%8C%80%EA%B8%B0%EC%98%A4%EC%97%BC_%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%99%80_%EB%AF%B8%EC%84%B8%EB%A8%BC%EC%A7%80%EC%9D%98_%EC%97%B0%EA%B4%80%EC%84%B1_%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request
import urllib.parse
import datetime
import json
import pandas as pd

In [ ]:
from google.colab import userdata
ServiceKey = userdata.get('fine_dust')

1. 표준 API 요청 함수

In [ ]:
## [CODE 1]
def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

2. 대기오염 정보 API 요청 및 JSON 파싱 함수

In [ ]:
## [CODE 2]
def reqAirInfo(sido, beginDay, endDay, numOfRows):
    service_url = 'http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getCtprvnRltmMesureDnsty'
    parameters = "?returnType=json&serviceKey=" + ServiceKey
    parameters += "&sidoName=" + urllib.parse.quote(sido)
    parameters += "&numOfRows=" + str(numOfRows)
    parameters += "&pageNo=1"
    parameters += "&ver=1.0"
    url = service_url + parameters
    print("API URL:", url)
    responseDecode = getRequestUrl(url)
    if (responseDecode == None):
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError as e:
            print("JSON 파싱 오류:", e)
            print("응답 내용:", responseDecode[:500])
            return None

3. 측정소별 실시간 대기 질 정보 요청 함수

In [ ]:
def reqStationAirInfo(stationName, numOfRows):
    service_url = 'http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty'
    parameters = "?returnType=json&serviceKey=" + ServiceKey
    parameters += "&stationName=" + urllib.parse.quote(stationName)
    parameters += "&dataTerm=DAILY"
    parameters += "&numOfRows=" + str(numOfRows)
    parameters += "&pageNo=1"
    parameters += "&ver=1.0"
    url = service_url + parameters
    print("측정소별 API URL:", url)
    responseDecode = getRequestUrl(url)
    if (responseDecode == None):
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError as e:
            print("JSON 파싱 오류:", e)
            print("응답 내용:", responseDecode[:500])
            return None

4. API 응답 데이터 항목 추출 및 정리 함수

In [ ]:
def getAirInfoItem(item, result):
    stationName = item.get('stationName', '')
    dataTime = item.get('dataTime', '')
    so2Value = item.get('so2Value', '')
    coValue = item.get('coValue', '')
    o3Value = item.get('o3Value', '')
    no2Value = item.get('no2Value', '')
    pm10Value = item.get('pm10Value', '')
    pm25Value = item.get('pm25Value', '')
    result.append([stationName, dataTime, so2Value, coValue, o3Value, no2Value, pm10Value, pm25Value])

In [ ]:
print("=== 대기오염 데이터 수집 ===")
print("1. 측정소별 조회")
print("2. 시도별 조회 (서울 전체)")
choice = input("선택하세요 (1 또는 2): ")

=== 대기오염 데이터 수집 ===
1. 측정소별 조회
2. 시도별 조회 (서울 전체)
선택하세요 (1 또는 2): 1


In [ ]:
if choice == "1":
    where = input('측정소명을 입력하세요 (예: 종로구): ')
    numOfRows = input('조회할 데이터 수 (예: 100): ')

    print(f"\n=== {where} 측정소 실시간 데이터 조회 ===")
    jsonResponse = reqStationAirInfo(where, numOfRows)

elif choice == "2":
    sido = input('시도명을 입력하세요 (예: 서울): ')
    numOfRows = input('조회할 데이터 수 (예: 100): ')

    print(f"\n=== {sido} 시도 전체 실시간 데이터 조회 ===")
    jsonResponse = reqAirInfo(sido, '', '', numOfRows)

else:
    print("잘못된 선택입니다.")
    exit()

측정소명을 입력하세요 (예: 종로구): 서초구
조회할 데이터 수 (예: 100): 20

=== 서초구 측정소 실시간 데이터 조회 ===
측정소별 API URL: http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty?returnType=json&serviceKey=RKp7mtfryeAiTFH74uWHBriQrfiZrYTEp1ujt5BSi9y7eqmJvOMV1dYp3KfFSp4uMf19WPjBjwz5vC%2Fe0UbDfg%3D%3D&stationName=%EC%84%9C%EC%B4%88%EA%B5%AC&dataTerm=DAILY&numOfRows=20&pageNo=1&ver=1.0
[2026-01-25 07:12:42.936207] Url Request Success


5. API 데이터 검증 및 데이터프레임 저장 로직

In [ ]:
result = []

if jsonResponse:
    print("API 응답 구조:", list(jsonResponse.keys()))

    if 'response' in jsonResponse:
        response = jsonResponse['response']

        if 'header' in response:
            header = response['header']
            result_code = header.get('resultCode')
            result_msg = header.get('resultMsg')
            print(f"응답 코드: {result_code}")
            print(f"응답 메시지: {result_msg}")

            if result_code == '00':
                print("API 호출 성공")
            elif result_code == '03':
                print("데이터가 없습니다.")
                exit()
            elif result_code == '11':
                print("필수 매개변수 누락")
                exit()
            elif result_code == '20':
                print("서비스 접근 권한 없음 - 공공데이터포털에서 서비스 신청 필요")
                exit()

        if 'body' in response and response['body']:
            body = response['body']

            if 'items' in body and body['items']:
                items = body['items']
                print(f"총 {len(items)}개의 실시간 데이터 발견")

                for item in items:
                    getAirInfoItem(item, result)

                if result:
                    columnNames = ["location", "datetime", "so2", "co", "o3", "no2", "pm10", "pm25"]
                    result_df = pd.DataFrame(result, columns=columnNames)

                    current_time = datetime.datetime.now().strftime('%Y%m%d_%H%M')
                    if choice == "1":
                        filename = f'대기오염데이터_{where}_{current_time}.csv'
                    else:
                        filename = f'대기오염데이터_{sido}전체_{current_time}.csv'

                    result_df.to_csv(filename, index=False, encoding='utf-8-sig')

                    print(f'\n파일 생성 완료: {filename}')
                    print(f"총 {len(result)}개의 데이터 저장됨")
                    print("\n데이터 미리보기:")
                    print(result_df.head(10))

                    if len(result_df) > 0:
                        print(f"\n측정소 현황:")
                        station_count = result_df['location'].value_counts()
                        print(station_count)
                else:
                    print("추출된 데이터가 없습니다.")
            else:
                print("응답에 items가 없습니다.")
                print("Body 내용:", body)
        else:
            print("응답에 body가 없습니다.")
    else:
        print("잘못된 응답 형식입니다.")
else:
    print("API 응답을 받지 못했습니다.")

print("\n=== 완료 ===")

API 응답 구조: ['response']
응답 코드: 00
응답 메시지: NORMAL_CODE
API 호출 성공
총 20개의 실시간 데이터 발견

파일 생성 완료: 대기오염데이터_서초구_20260125_0712.csv
총 20개의 데이터 저장됨

데이터 미리보기:
  location          datetime    so2   co     o3    no2 pm10 pm25
0           2026-01-25 16:00  0.003  0.3  0.036  0.009   23   11
1           2026-01-25 15:00  0.003  0.3  0.034  0.010   29   11
2           2026-01-25 14:00  0.003  0.3  0.032  0.011   26    8
3           2026-01-25 13:00  0.003  0.4  0.033  0.010   22   11
4           2026-01-25 12:00  0.003  0.3  0.031  0.011   23   11
5           2026-01-25 11:00  0.003  0.4  0.029  0.012   22    9
6           2026-01-25 10:00  0.003  0.4  0.030  0.011   22   13
7           2026-01-25 09:00  0.003  0.4  0.028  0.013   23    9
8           2026-01-25 08:00  0.003  0.3  0.027  0.013   23   11
9           2026-01-25 07:00  0.003  0.4  0.027  0.013   21   13

측정소 현황:
location
    20
Name: count, dtype: int64

=== 완료 ===


6. 데이터프레임 생성 및 특정 컬럼 정보 수정

In [ ]:
import pandas as pd

data_df=pd.DataFrame(result, columns=["location", "day", "so2", "co", "o3", "no2", "pm10", "pm25"])
data_df = data_df.drop('location', axis=1)
data_df.insert(0, 'location', '종로구')
data_df

,location,day,so2,co,o3,no2,pm10,pm25
0,종로구,2026-01-25 16:00,0.003,0.3,0.036,0.009,23,11
1,종로구,2026-01-25 15:00,0.003,0.3,0.034,0.010,29,11
2,종로구,2026-01-25 14:00,0.003,0.3,0.032,0.011,26,8
3,종로구,2026-01-25 13:00,0.003,0.4,0.033,0.010,22,11
4,종로구,2026-01-25 12:00,0.003,0.3,0.031,0.011,23,11
5,종로구,2026-01-25 11:00,0.003,0.4,0.029,0.012,22,9
6,종로구,2026-01-25 10:00,0.003,0.4,0.030,0.011,22,13
7,종로구,2026-01-25 09:00,0.003,0.4,0.028,0.013,23,9
8,종로구,2026-01-25 08:00,0.003,0.3,0.027,0.013,23,11
9,종로구,2026-01-25 07:00,0.003,0.4,0.027,0.013,21,13


In [ ]:
import pandas as pd

data_df1=pd.DataFrame(result, columns=["location", "day", "so2", "co", "o3", "no2", "pm10", "pm25"])
data_df1 = data_df1.drop('location', axis=1)
data_df1.insert(0, 'location', '서초구')
data_df1

,location,day,so2,co,o3,no2,pm10,pm25
0,서초구,2026-01-25 16:00,0.003,0.3,0.036,0.009,23,11
1,서초구,2026-01-25 15:00,0.003,0.3,0.034,0.010,29,11
2,서초구,2026-01-25 14:00,0.003,0.3,0.032,0.011,26,8
3,서초구,2026-01-25 13:00,0.003,0.4,0.033,0.010,22,11
4,서초구,2026-01-25 12:00,0.003,0.3,0.031,0.011,23,11
5,서초구,2026-01-25 11:00,0.003,0.4,0.029,0.012,22,9
6,서초구,2026-01-25 10:00,0.003,0.4,0.030,0.011,22,13
7,서초구,2026-01-25 09:00,0.003,0.4,0.028,0.013,23,9
8,서초구,2026-01-25 08:00,0.003,0.3,0.027,0.013,23,11
9,서초구,2026-01-25 07:00,0.003,0.4,0.027,0.013,21,13
